In [1]:
import pandas as pd
import numpy as np
import statistics
from pyprojroot import here

In [2]:
INTERIM_DATA_DIR = here() / "data" / "interim"
PROCESSED_DATA_DIR = here() / "data" / "processed"

In [3]:
norm = pd.read_csv(INTERIM_DATA_DIR/"master_norm.csv", encoding="utf-8")

### Sub Indexes calculation

In [4]:
norm['idx_resources'] = norm[
    ['pv_mean_annual_kwh',
     'wind_mean',
     'water_availability',
     'rainfall']
].mean(axis=1)

norm['idx_energy'] = norm[
    ['dist_min_km',
    'gen_min_km']
].mean(axis=1)

norm['idx_logistics'] = norm[
    ['distance_ciit',
     'dist_to_border_km',
     'dist_min_ports',
     'rails',
     'highways']
].mean(axis=1)

norm['idx_market'] = norm[
    ['n_industy',
     'economic_output']
].mean(axis=1)

norm['idx_security'] = norm[
    ['crime']
].mean(axis=1)

# R.E.L.M.S. index 

In [5]:
subindex_cols_RELMS = [
    'idx_resources',
    'idx_energy',
    'idx_logistics',
    'idx_market',
    'idx_security'
]

weights_RELMS = np.repeat(
    1 / len(subindex_cols_RELMS),
    len(subindex_cols_RELMS)
)

In [6]:
eps = 1e-10

norm['RELMS_index'] = np.exp(
    (
        weights_RELMS *
        np.log(
            norm[subindex_cols_RELMS].values + eps
        )
    ).sum(axis=1)
)

In [7]:
norm.sort_values(by='RELMS_index')

,Unnamed: 0,CVEGEO,water_availability,economic_output,n_industy,distance_ciit,dist_to_border_km,dist_min_ports,highways,dist_min_km,...,pv_mean_annual_kwh,wind_mean,rainfall,crime,idx_resources,idx_energy,idx_logistics,idx_market,idx_security,RELMS_index
2422,2422,32027,0.000000,NaN,0.000000,0.184485,0.696288,0.098224,1.0,0.150671,...,0.826748,0.590030,0.712544,1.000000,0.532330,0.172537,0.395799,0.000000,1.000000,0.005154
2039,2039,28036,0.357214,NaN,0.000000,0.207505,0.863553,0.220066,1.0,0.165199,...,0.543639,0.619504,0.000000,0.925715,0.380089,0.229360,0.458225,0.000000,0.925715,0.005171
1426,1426,20288,0.453268,NaN,0.000000,0.496595,0.561823,0.289441,0.0,0.307668,...,0.334781,0.004882,0.869453,1.000000,0.415596,0.344322,0.269572,0.000000,1.000000,0.005215
1498,1498,20408,0.000000,NaN,0.000000,0.406050,0.458445,0.220458,1.0,0.355543,...,0.821103,0.472175,0.000000,1.000000,0.323320,0.306371,0.416991,0.000000,1.000000,0.005287
1886,1886,24030,0.252031,NaN,0.000000,0.242249,0.592266,0.217884,1.0,0.301032,...,0.590534,0.137589,0.760646,0.709778,0.435200,0.342321,0.410480,0.000000,0.709778,0.005340
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2339,2339,31101,1.000000,0.758597,0.604688,0.251970,0.414229,0.532212,1.0,0.474055,...,0.627877,0.674983,0.821292,0.604663,0.781038,0.539279,0.639682,0.681642,0.604663,0.644323
1325,1325,20014,0.393220,0.406341,0.663224,1.000000,0.695775,0.525973,1.0,0.614836,...,0.627008,0.836376,0.821292,0.589684,0.669474,0.635193,0.844350,0.534783,0.589684,0.646832
1082,1082,20124,0.393220,0.437450,0.785407,0.795166,0.701639,0.687174,1.0,0.489820,...,0.801129,0.876077,0.821292,0.717172,0.722930,0.491018,0.836796,0.611429,0.717172,0.665206
1020,1020,20441,0.393220,0.398404,0.760024,0.794740,0.711249,0.639137,1.0,0.604923,...,0.829787,1.000000,0.821292,1.000000,0.761075,0.548407,0.629025,0.579214,1.000000,0.686132


# R.E.L.M. index

In [8]:
subindex_cols_RELM = [
    'idx_resources',
    'idx_energy',
    'idx_logistics',
    'idx_market'
]

weights_RELM = np.repeat(
    1 / len(subindex_cols_RELM),
    len(subindex_cols_RELM)
)

### Ranking by R.E.L.M.S.

In [9]:
norm['RELM_index'] = np.exp(
    (
        weights_RELM *
        np.log(
            norm[subindex_cols_RELM].values + eps
        )
    ).sum(axis=1)
)

In [10]:
norm['rank'] = (
    norm['RELMS_index']
    .rank(ascending=False, method='dense')
    .astype(int)
)

In [11]:
norm = norm.sort_values(
    by='RELMS_index',
    ascending=False
)

### Export indexes to CSV

In [12]:
norm.to_csv(PROCESSED_DATA_DIR/
    "RELMS_index.csv",
    index=False,
    encoding="utf-8"
)

print(norm[
    ['CVEGEO',
     'idx_resources',
     'idx_energy',
     'idx_logistics',
     'idx_market',
     'idx_security',
     'RELMS_index',
     'rank',
     'RELM_index'
    ]
].head(20))

      CVEGEO  idx_resources  idx_energy  idx_logistics  idx_market  \
1581   20557       0.743142    0.646419       0.799718    0.481783   
1020   20441       0.761075    0.548407       0.629025    0.579214   
1082   20124       0.722930    0.491018       0.836796    0.611429   
1325   20014       0.669474    0.635193       0.844350    0.534783   
2339   31101       0.781038    0.539279       0.639682    0.681642   
2099   29043       0.690144    0.712129       0.603378    0.543131   
2098   29038       0.676141    0.580943       0.606267    0.570894   
2091   29005       0.694878    0.630627       0.605349    0.727237   
2238   30059       0.643488    0.513268       0.754356    0.583167   
2046   29060       0.510536    0.675736       0.600673    0.499689   
2078   29048       0.512789    0.724628       0.601338    0.585497   
2090   29039       0.653452    0.510264       0.608626    0.616531   
2344   31050       0.779751    0.751339       0.649187    0.864460   
2044   29059       0

In [13]:
df_subindx = norm[
    [
        'CVEGEO',
        'idx_resources',
        'idx_energy',
        'idx_logistics',
        'idx_market',
        'idx_security'
    ]
]

df_subindx.to_csv(PROCESSED_DATA_DIR/
    "RELMS_subindex.csv",
    index=False
)